# ClearGuide Traffic Impact Portfolio Notebook

This notebook presents a portfolio-ready version of a corridor-level traffic impact analysis conducted after the Francis Scott Key Bridge collapse in Baltimore.

## What this notebook shows
- how high-frequency corridor data were cleaned and filtered
- how immediate and seasonal effects were compared
- how the most severely impacted routes were identified
- how fixed-effects and mixed-effects models were used to move beyond simple descriptive comparisons

## Data note
The source dataset is not included in this repository. To run this notebook, place the prepared file at:

`../data/Merged_Modeling_Ready_with_Proximity_Level.csv`

## 1. Setup
The code below loads the dataset and standardizes basic time fields used throughout the analysis.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.formula.api import mixedlm

from src.analysis_utils import (
    IMMEDIATE,
    FALL,
    WINTER,
    add_peak_column,
    filter_two_period_window,
    load_and_prepare_data,
    top_impacted_routes,
)

sns.set_theme(style="whitegrid")

DATA_PATH = Path("../data/Merged_Modeling_Ready_with_Proximity_Level.csv")
OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

df = load_and_prepare_data(DATA_PATH)
df = add_peak_column(df)

print(df.shape)
df.head()

## 2. Framing the event window

For the immediate analysis, the baseline and impact windows are defined as:

- **Before:** February 26, 2024 to March 25, 2024
- **After:** March 26, 2024 to April 25, 2024

The analysis focuses on peak-period traffic conditions using Travel Time Index (`tti_freeflow`).

In [ ]:
df_immediate = filter_two_period_window(df, IMMEDIATE)
df_immediate = df_immediate[df_immediate["Peak"].isin(["AM Peak", "PM Peak"])].copy()

summary_counts = (
    df_immediate.groupby(["Period", "Peak"])
    .size()
    .reset_index(name="records")
)

summary_counts

## 3. Immediate before/after comparison

This section gives the fastest stakeholder-facing read of the event:
- Did TTI worsen?
- Was the effect stronger in the AM or PM peak?
- How quickly did conditions change after the collapse?

In [ ]:
mean_tti_immediate = (
    df_immediate.groupby(["Peak", "Period"])["tti_freeflow"]
    .mean()
    .reset_index()
)

mean_tti_immediate

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=mean_tti_immediate, x="Peak", y="tti_freeflow", hue="Period")
plt.title("Immediate Scenario: Mean Peak-Period TTI Before vs. After Collapse")
plt.ylabel("Mean TTI")
plt.xlabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "immediate_peak_tti_before_after.png", dpi=200)
plt.show()

In [ ]:
daily_immediate = (
    df_immediate.groupby(["Date", "Peak"])["tti_freeflow"]
    .mean()
    .reset_index()
)

for peak_label in ["AM Peak", "PM Peak"]:
    plot_df = daily_immediate[daily_immediate["Peak"] == peak_label]
    plt.figure(figsize=(10, 4))
    plt.plot(plot_df["Date"], plot_df["tti_freeflow"], marker="o")
    plt.axvline(pd.Timestamp("2024-03-26"), linestyle="--")
    plt.title(f"{peak_label}: Daily Mean TTI During Immediate Study Window")
    plt.xlabel("")
    plt.ylabel("Mean TTI")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"daily_tti_{peak_label.lower().replace(' ', '_')}.png", dpi=200)
    plt.show()

## 4. Route severity stratification

A simple systemwide average can hide where the disruption was most severe.

To address that, the original workflow identified the **top 20% most impacted routes** based on PM-peak change in mean TTI. This creates a high-impact subset for more targeted modeling.

In [ ]:
df_immediate_pm = df_immediate[df_immediate["Peak"] == "PM Peak"].copy()
severe_routes = top_impacted_routes(df_immediate_pm, quantile=0.80)

len(severe_routes), severe_routes[:10]

In [ ]:
df_top20 = df_immediate[df_immediate["Route_Number"].isin(severe_routes)].copy()

top20_means = (
    df_top20.groupby(["Peak", "Period"])["tti_freeflow"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(8, 5))
sns.barplot(data=top20_means, x="Peak", y="tti_freeflow", hue="Period")
plt.title("Top 20% Impacted Routes: Mean TTI Before vs. After")
plt.ylabel("Mean TTI")
plt.xlabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "top20_routes_before_after.png", dpi=200)
plt.show()

## 5. Fixed-effects-style specification

The first model estimates how TTI changed by period and peak while adjusting for route proximity categories.

Interpretation focus:
- the **After** coefficient captures the overall post-collapse shift
- the **Period × Peak** interaction checks whether AM and PM moved differently
- proximity coefficients show whether certain route groups were structurally worse than the reference category

In [ ]:
fe_df = df_immediate.copy()
fe_df["Period"] = fe_df["Period"].astype("category")
fe_df["Peak"] = fe_df["Peak"].astype("category")
fe_df["Proximity_Level"] = fe_df["Proximity_Level"].astype("category")

fe_model = smf.ols(
    "tti_freeflow ~ Period * Peak + C(Proximity_Level)",
    data=fe_df
).fit()

fe_results = pd.DataFrame({
    "Term": fe_model.params.index,
    "Estimate": fe_model.params.values,
    "Std_Error": fe_model.bse.values,
    "t_value": fe_model.tvalues.values,
    "p_value": fe_model.pvalues.values,
    "CI_Lower": fe_model.conf_int()[0].values,
    "CI_Upper": fe_model.conf_int()[1].values,
})

fe_results.to_csv(OUTPUT_DIR / "immediate_fixed_effects_results.csv", index=False)
fe_results

In [ ]:
prox_df = fe_results[fe_results["Term"].str.contains("C\(Proximity_Level\)")].copy()
prox_df["Level"] = prox_df["Term"].str.extract(r"\[T\.(.*)\]")

prox_df = pd.concat([
    pd.DataFrame([{
        "Term": "Reference",
        "Estimate": 0.0,
        "Std_Error": None,
        "t_value": None,
        "p_value": None,
        "CI_Lower": 0.0,
        "CI_Upper": 0.0,
        "Level": "Far"
    }]),
    prox_df
], ignore_index=True)

plt.figure(figsize=(8, 5))
sns.barplot(data=prox_df, y="Level", x="Estimate", orient="h")
for i, row in prox_df.reset_index(drop=True).iterrows():
    plt.plot([row["CI_Lower"], row["CI_Upper"]], [i, i], linewidth=1)
plt.axvline(0, linestyle="--")
plt.title("Estimated TTI Difference by Proximity Level")
plt.xlabel("Difference vs. Far Reference Group")
plt.ylabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "proximity_effects.png", dpi=200)
plt.show()

## 6. Stratified model for the most disrupted routes

The next model applies the same logic only to the high-impact corridors. This is useful when the planning question is not just “Did traffic worsen?” but “Where did it worsen enough to matter operationally?”

In [ ]:
strat_df = df_top20.copy()
strat_df["Period"] = strat_df["Period"].astype("category")
strat_df["Peak"] = strat_df["Peak"].astype("category")
strat_df["Proximity_Level"] = strat_df["Proximity_Level"].astype("category")

strat_model = smf.ols(
    "tti_freeflow ~ Period * Peak + C(Proximity_Level)",
    data=strat_df
).fit()

strat_results = pd.DataFrame({
    "Term": strat_model.params.index,
    "Estimate": strat_model.params.values,
    "Std_Error": strat_model.bse.values,
    "t_value": strat_model.tvalues.values,
    "p_value": strat_model.pvalues.values,
    "CI_Lower": strat_model.conf_int()[0].values,
    "CI_Upper": strat_model.conf_int()[1].values,
})

strat_results.to_csv(OUTPUT_DIR / "top20_fixed_effects_results.csv", index=False)
strat_results

## 7. Mixed-effects model

Repeated observations from the same route are not independent. A mixed-effects model helps account for that structure by:
- adding a random effect for `Route_Number`
- allowing weekday-related variance to be modeled separately

This is a stronger portfolio signal than a pure before/after chart because it shows awareness of hierarchical traffic data structure.

In [ ]:
re_df = df_immediate_pm[df_immediate_pm["Route_Number"].isin(severe_routes)].copy()
re_df["Period"] = re_df["Period"].astype("category")
re_df["Proximity_Level"] = re_df["Proximity_Level"].astype("category")
re_df["Weekday"] = re_df["Weekday"].astype("category")

re_model = mixedlm(
    "tti_freeflow ~ Period + C(Proximity_Level)",
    data=re_df,
    groups=re_df["Route_Number"],
    vc_formula={"Weekday": "0 + C(Weekday)"}
)
re_result = re_model.fit()

re_results = pd.DataFrame({
    "Term": re_result.fe_params.index,
    "Estimate": re_result.fe_params.values,
    "Std_Error": re_result.bse_fe.values,
    "z_value": (re_result.fe_params / re_result.bse_fe).values,
    "p_value": re_result.pvalues.iloc[:len(re_result.fe_params)].values,
    "CI_Lower": re_result.conf_int().loc[re_result.fe_params.index, 0].values,
    "CI_Upper": re_result.conf_int().loc[re_result.fe_params.index, 1].values,
})

re_results.to_csv(OUTPUT_DIR / "top20_mixed_effects_results.csv", index=False)
re_results

## 8. Seasonal robustness checks

The immediate spike is important, but a stronger resilience story asks whether post-collapse effects persisted into later seasonal comparisons.

This section compares:
- Fall 2023 vs. Fall 2024
- Winter 2023–24 vs. Winter 2024–25

In [ ]:
def seasonal_mean_table(df, scenario):
    temp = filter_two_period_window(df, scenario)
    temp = temp[temp["Peak"].isin(["AM Peak", "PM Peak"])].copy()
    return (
        temp.groupby(["Peak", "Period"])["tti_freeflow"]
        .mean()
        .reset_index()
        .assign(Scenario=scenario.name)
    )

seasonal_summary = pd.concat([
    seasonal_mean_table(df, FALL),
    seasonal_mean_table(df, WINTER)
], ignore_index=True)

seasonal_summary

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=seasonal_summary, x="Scenario", y="tti_freeflow", hue="Period")
plt.title("Seasonal Robustness Check: Mean TTI Before vs. After")
plt.ylabel("Mean TTI")
plt.xlabel("")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "seasonal_robustness_summary.png", dpi=200)
plt.show()

In [ ]:
def run_seasonal_top20_model(df, scenario):
    season_df = filter_two_period_window(df, scenario)
    season_df = season_df[season_df["Peak"] == "PM Peak"].copy()
    top_routes = top_impacted_routes(season_df, quantile=0.80)
    season_top = season_df[season_df["Route_Number"].isin(top_routes)].copy()
    season_top["Period"] = season_top["Period"].astype("category")
    season_top["Proximity_Level"] = season_top["Proximity_Level"].astype("category")

    model = smf.ols("tti_freeflow ~ Period + C(Proximity_Level)", data=season_top).fit()
    out = pd.DataFrame({
        "Term": model.params.index,
        "Estimate": model.params.values,
        "Std_Error": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
        "CI_Lower": model.conf_int()[0].values,
        "CI_Upper": model.conf_int()[1].values,
        "Scenario": scenario.name,
    })
    return out

seasonal_model_results = pd.concat([
    run_seasonal_top20_model(df, FALL),
    run_seasonal_top20_model(df, WINTER)
], ignore_index=True)

seasonal_model_results.to_csv(OUTPUT_DIR / "seasonal_top20_model_results.csv", index=False)
seasonal_model_results

## 9. What this portfolio piece demonstrates

### Technical skills
- time-series traffic data preparation
- peak-period filtering and scenario design
- route stratification logic
- regression and mixed-effects modeling
- conversion of research analysis into stakeholder-facing outputs

### Transportation skills
- Travel Time Index interpretation
- corridor performance monitoring
- resilience and disruption framing
- operational relevance beyond generic machine learning

## Final publishing tips
Before pushing to GitHub:
1. add 2 to 4 exported figures under `figures/`
2. replace placeholder outputs with your final rerun outputs
3. add a short “Results highlights” section in the README with exact values
4. pin one clean screenshot of the notebook or charts at the top of the repository